In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path('../datasets').resolve()

DATASETS = [
    {'name': 'adult',              'folder': 'adult',              'file_prefix': 'adult',              'target': 'income',             'task': 'classification'},
    {'name': 'bank-marketing',     'folder': 'bank-marketing',     'file_prefix': 'bank_marketing',     'target': 'deposit',            'task': 'classification'},
    {'name': 'california-housing', 'folder': 'california-housing', 'file_prefix': 'california_housing', 'target': 'median_house_value', 'task': 'regression'},
    {'name': 'cardiotocography',   'folder': 'cardiotocography',   'file_prefix': 'cardiotocography',   'target': 'CLASS',              'task': 'classification'},
    {'name': 'compas',             'folder': 'compas',             'file_prefix': 'compas',             'target': 'is_recid',           'task': 'classification'},
    {'name': 'loan-predication',   'folder': 'loan-predication',   'file_prefix': 'loan_predication',   'target': 'Loan_Status',        'task': 'classification'},
]

SEED = 42

In [ ]:
def load_full(cfg, seed=SEED):
    train = pd.read_csv(BASE / cfg['folder'] / 'data' / f"{cfg['file_prefix']}_seed{seed}_train.csv")
    test  = pd.read_csv(BASE / cfg['folder'] / 'data' / f"{cfg['file_prefix']}_seed{seed}_test.csv")
    return pd.concat([train, test], ignore_index=True)

def summarise(cfg):
    df = load_full(cfg)
    target = cfg['target']
    features = df.drop(columns=[target])
    cat_cols = features.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    num_cols = features.select_dtypes(include='number').columns.tolist()
    n_classes = df[target].nunique() if cfg['task'] == 'classification' else None
    task_detail = cfg['task']
    if cfg['task'] == 'classification':
        task_detail = 'binary classification' if n_classes == 2 else 'multiclass classification'
    return {
        'dataset': cfg['name'],
        'n_rows': len(df),
        'n_columns': df.shape[1],
        'task': task_detail,
        'target': target,
        'n_categorical': len(cat_cols),
        'n_numerical': len(num_cols),
        'n_classes': n_classes,
    }

summary = pd.DataFrame([summarise(c) for c in DATASETS])
summary

In [ ]:
out = BASE / 'datasets_summary.csv'
summary.to_csv(out, index=False)
print(f'Saved to {out}')